In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import librosa
import numpy as np
from tqdm import tqdm

# STFT settings
SR = 16000
N_FFT = 512
HOP = 128  # 8ms at 16kHz
WIN = 512



TARGET_FRAMES = 256   # fixed time length

In [ ]:
def compute_stft(audio):
    stft = librosa.stft(
        audio,
        n_fft=N_FFT,
        hop_length=HOP,
        win_length=WIN,
        window="hann",
        center=True
    )
    mag = np.log1p(np.abs(stft))   # LOG magnitude
    phase = np.angle(stft)
    return mag, phase

In [ ]:
def fix_length(spec, target_frames):
    """
    spec: (freq_bins, time_frames)
    """
    T = spec.shape[1]

    if T > target_frames:
        return spec[:, :target_frames]
    else:
        pad = target_frames - T
        return np.pad(spec, ((0, 0), (0, pad)), mode="constant")


In [ ]:
clean_audio_dir = "/content/drive/MyDrive/ML_Project/mixed_dataset/clean"
noisy_audio_dir = "/content/drive/MyDrive/ML_Project/mixed_dataset/noisy"

clean_out = "/content/drive/MyDrive/ML_Project/spectrogram_dataset/clean_mag"
noisy_out = "/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_mag"
phase_out = "/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase"

os.makedirs(clean_out, exist_ok=True)
os.makedirs(noisy_out, exist_ok=True)
os.makedirs(phase_out, exist_ok=True)

In [ ]:
clean_ids = {os.path.splitext(f)[0] for f in os.listdir(clean_audio_dir)}
noisy_ids = {os.path.splitext(f)[0] for f in os.listdir(noisy_audio_dir)}

In [ ]:
clean_only = sorted(clean_ids - noisy_ids)

print("Clean files with NO noisy match:")
for f in clean_only:
    print(f)


Clean files with NO noisy match:


In [ ]:
noisy_only = sorted(noisy_ids - clean_ids)

print("\nNoisy files with NO clean match:")
for f in noisy_only:
    print(f)



Noisy files with NO clean match:


In [ ]:
clean_files = sorted(os.listdir(clean_audio_dir))
noisy_files = sorted(os.listdir(noisy_audio_dir))

print(len(clean_files))
print(len(noisy_files))
assert len(clean_files) == len(noisy_files)

for clean_f, noisy_f in tqdm(zip(clean_files, noisy_files), total=len(clean_files)):
    clean_path = os.path.join(clean_audio_dir, clean_f)
    noisy_path = os.path.join(noisy_audio_dir, noisy_f)

    # Load audio
    clean_audio, _ = librosa.load(clean_path, sr=SR)
    noisy_audio, _ = librosa.load(noisy_path, sr=SR)

    # STFT
    clean_mag, _ = compute_stft(clean_audio)
    noisy_mag, noisy_phase = compute_stft(noisy_audio)

    # Fix time length (NO interpolation!)
    clean_mag = fix_length(clean_mag, TARGET_FRAMES)
    noisy_mag = fix_length(noisy_mag, TARGET_FRAMES)
    noisy_phase = fix_length(noisy_phase, TARGET_FRAMES)

    # Save
    name = os.path.splitext(clean_f)[0] + ".npy"
    np.save(os.path.join(clean_out, name), clean_mag.astype(np.float32))
    np.save(os.path.join(noisy_out, name), noisy_mag.astype(np.float32))
    np.save(os.path.join(phase_out, name), noisy_phase.astype(np.float32))


  0%|          | 0/8923 [00:00<?, ?it/s]

In [ ]:
primt